# Práctica: Asistente de Código Personalizado con Redes Recurrentes (RNN Vanilla)

Este notebook contiene la implementación para entrenar un modelo a nivel de caracteres utilizando una **RNN Vanilla (SimpleRNN)** en Keras/TensorFlow. El objetivo es que el modelo aprenda el estilo de programación de C a partir de las 65 funciones personalizadas en `dataset_funciones_c.txt` y sea capaz de autocompletar código en tiempo real.

## 1. Importación de Librerías y Carga del Dataset

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path
import json

# Fijar semillas para reproducibilidad
tf.keras.utils.set_random_seed(42)

# Cargar el dataset de funciones en C
dataset_path = Path("dataset_funciones_c.txt")
if not dataset_path.exists():
    raise FileNotFoundError(f"No se encontró el archivo {dataset_path}. Por favor, asegúrate de que esté en el mismo directorio.")

with open(dataset_path, "r", encoding="utf-8") as f:
    CORPUS = f.read()

print(f"Longitud del corpus: {len(CORPUS)} caracteres.")
print("--- Primeros 300 caracteres del corpus ---")
print(CORPUS[:300])

## 2. Creación del Vocabulario y Funciones de Codificación

Mapeamos cada carácter único a un índice numérico para poder alimentar a la red neuronal.

In [ ]:
# Caracteres únicos en el corpus
chars = sorted(list(set(CORPUS)))
VOCAB_SIZE = len(chars)

# Diccionarios de mapeo
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]

def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)

print(f"Tamaño del vocabulario: {VOCAB_SIZE} caracteres únicos.")
print("Vocabulario:", repr("".join(chars)))

## 3. Preparación de las Ventanas de Entrenamiento (Secuencias)

Creamos pares de secuencias `X` (entrada) e `Y` (salida desplazada por un carácter) con un tamaño de ventana (`BLOCK_SIZE`) fijo.

In [ ]:
BLOCK_SIZE = 64

# Codificar todo el corpus
SEQ = np.array(encode(CORPUS), dtype=np.int64)

X_rows, Y_rows = [], []
for i in range(0, len(SEQ) - BLOCK_SIZE):
    X_rows.append(SEQ[i : i + BLOCK_SIZE])
    Y_rows.append(SEQ[i + 1 : i + 1 + BLOCK_SIZE])

X = np.stack(X_rows)
Y = np.stack(Y_rows)

print(f"Forma de X: {X.shape} (N_secuencias, longitud_secuencia)")
print(f"Forma de Y: {Y.shape}")
print("\n--- Ejemplo de par de entrenamiento ---")
print("Entrada (X[0]):", repr(decode(X[0])))
print("Objetivo (Y[0]):", repr(decode(Y[0])))

## 4. Construcción del Modelo Keras (SimpleRNN)

Diseñamos el modelo secuencial con una capa `Embedding`, una capa `SimpleRNN` recurrente pura, y una capa `TimeDistributed` con una Densa de salida del tamaño del vocabulario.

In [ ]:
EMBED_DIM = 64
HIDDEN = 256

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(BLOCK_SIZE,)),
    tf.keras.layers.Embedding(VOCAB_SIZE, EMBED_DIM),
    tf.keras.layers.LSTM(HIDDEN, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(VOCAB_SIZE)),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

model.summary()

## 5. Entrenamiento del Modelo

Entrenamos el modelo. Al ser un corpus mediano, se puede entrenar en CPU con un tamaño de lote pequeño para ver cómo baja la pérdida.

In [ ]:
EPOCHS = 150
BATCH_SIZE = 32

print("Iniciando entrenamiento...")
history = model.fit(
    X,
    Y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print(f"Pérdida inicial: {history.history['loss'][0]:.4f}")
print(f"Pérdida final: {history.history['loss'][-1]:.4f}")

## 6. Generación Autoregresiva y Autocompletado

Implementamos funciones para predecir el siguiente carácter a partir de un prefijo dado, utilizando un muestreo probabilístico con ajuste de temperatura.

In [ ]:
def complete(prompt: str, max_new: int = 120, temperature: float = 0.2) -> str:
    ids = encode(prompt)
    rng = np.random.default_rng(42)
    for _ in range(max_new):
        # Tomar los últimos caracteres como contexto de longitud BLOCK_SIZE
        x = np.array(ids[-BLOCK_SIZE:], dtype=np.int64)
        if x.shape[0] < BLOCK_SIZE:
            # Rellenar a la izquierda si el prompt es más corto
            pad = np.full(BLOCK_SIZE - x.shape[0], ids[0], dtype=np.int64)
            x = np.concatenate([pad, x])
        x = x.reshape(1, BLOCK_SIZE)
        
        # Obtener logits de la última posición
        logits = model(x, training=False).numpy()[0, -1, :]
        logits = logits / max(temperature, 1e-6)
        logits = logits - logits.max()  # Estabilidad numérica
        probs = np.exp(logits)
        probs = probs / probs.sum()
        
        # Muestrear el siguiente índice de carácter
        next_id = int(rng.choice(len(probs), p=probs))
        ids.append(next_id)
        
    return decode(ids)

# Prueba de completado
prompt_prueba = "// [CRIS_CODE]\nint SUMAR_ENTEROS(int"
resultado = complete(prompt_prueba, max_new=80, temperature=0.2)
print("--- Resultado de Completado ---")
print(resultado)

## 7. Guardar el Modelo para el Servidor

Exportamos el modelo entrenado en formato Keras y un archivo de configuración JSON con el vocabulario, de modo que el servidor JSON stdio pueda consumirlo.

In [ ]:
DEPLOY_DIR = Path("rnn-keras-autocomplete")
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

# Guardar el modelo en formato .keras
model.save(DEPLOY_DIR / "model.keras")

# Guardar metadatos (vocabulario y block_size)
meta_data = {
    "block_size": BLOCK_SIZE,
    "chars": chars
}

with open(DEPLOY_DIR / "meta.json", "w", encoding="utf-8") as json_file:
    json.dump(meta_data, json_file, ensure_ascii=False)

print(f"¡Éxito! Modelo y metadatos guardados en la carpeta: '{DEPLOY_DIR.resolve()}'")